In [0]:
from pyspark.sql import functions as F

CATALOGO = "mvp"
ESQUEMA = "staging"

TABELA_BRONZE = f"{CATALOGO}.{ESQUEMA}.bronze_diabetes_raw"
TABELA_SILVER = f"{CATALOGO}.{ESQUEMA}.silver_diabetes_clean"

df_bronze = spark.table(TABELA_BRONZE)

# Padronizar nomes de colunas para minúsculas
for c in df_bronze.columns:
    df_bronze = df_bronze.withColumnRenamed(c, c.strip().lower())

colunas = df_bronze.columns

# Mapeamento flexível de nomes de colunas
def obter_coluna(opcoes):
    for op in opcoes:
        if op in colunas:
            return op
    return None

col_glicose   = obter_coluna(["fastingbloodsugar", "fasting_blood_sugar", "glucose", "glicemia"])
col_historico = obter_coluna(["familyhistorydiabetes", "family_history_diabetes", "historico_familiar"])
col_atividade = obter_coluna(["physicalactivitylevel", "physical_activity_level", "atividade_fisica"])
col_diabetes  = obter_coluna(["diabetes", "outcome", "target"])

# Transformações e Tratamento de Tipos
df_silver = df_bronze \
    .withColumn("age", F.col("age").cast("int")) \
    .withColumn("bmi", F.col("bmi").cast("double")) \
    .withColumn("fasting_blood_sugar", F.col(col_glicose).cast("double") if col_glicose else F.lit(100.0)) \
    .withColumn(
        "family_history_diabetes",
        F.when(F.lower(F.trim(F.col(col_historico).cast("string"))).isin(["yes", "true", "1"]), 1)
         .when(F.lower(F.trim(F.col(col_historico).cast("string"))).isin(["no", "false", "0"]), 0)
         .otherwise(0) if col_historico else F.lit(0)
    ) \
    .withColumn(
        "diabetes",
        F.when(F.lower(F.trim(F.col(col_diabetes).cast("string"))).isin(["yes", "true", "1"]), 1)
         .when(F.lower(F.trim(F.col(col_diabetes).cast("string"))).isin(["no", "false", "0"]), 0)
         .otherwise(0) if col_diabetes else F.lit(0)
    ) \
    .withColumn("physical_activity_level", F.lower(F.trim(F.col(col_atividade).cast("string"))) if col_atividade else F.lit("moderate")) \
    .dropDuplicates() \
    .filter(F.col("age").isNotNull() & F.col("bmi").isNotNull()) \
    .drop("_ingestion_datetime", "_source_file")

# Persistência
df_silver.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(TABELA_SILVER)

print(f"✓ Camada Silver criada com sucesso! Linhas: {spark.table(TABELA_SILVER).count()}")

In [0]:
%sql
-- Verificar se restou algum registro nulo ou duplicado
SELECT 
  COUNT(*) AS total_registros,
  SUM(CASE WHEN age IS NULL THEN 1 ELSE 0 END) AS nulos_idade,
  SUM(CASE WHEN bmi IS NULL THEN 1 ELSE 0 END) AS nulos_imc
FROM mvp.staging.silver_diabetes_clean;